# MOSTA: velocity + communication (Brain focus, t3)

This notebook reproduces the 4 communication-overlay velocity plots at MOSTA timepoint index `t3` (usually **E15.5**) for **focus = Brain**, in **physical + gene** spaces.


In [ ]:
# Setup
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt


def _find_downstream_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / 'downstream_helpers').is_dir() and (parent / 'vendor').is_dir():
            return parent
    return start


DOWNSTREAM_ROOT = _find_downstream_root(Path.cwd().resolve())
CYTOBRIDGE_REPO = (DOWNSTREAM_ROOT.parent / 'cytobridge-spatial').resolve()
VENDOR_ROOT = (DOWNSTREAM_ROOT / 'vendor').resolve()
for path in (DOWNSTREAM_ROOT, VENDOR_ROOT, CYTOBRIDGE_REPO):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print('Downstream root:', DOWNSTREAM_ROOT)
print('CytoBridge repo:', CYTOBRIDGE_REPO)

# cb_pipeline / DeepRUOTv2 env deps
import numpy as np
import pandas as pd

import torch
import anndata as ad
import scanpy as sc
import scvelo as scv

from evaluation.arista_code import arista_helpers as helpers
from evaluation.arista_code.mosta_ported.velocity import VelocityAnalyzer
from downstream_helpers import load_mosta_context

scv.settings.set_figure_params('scvelo', dpi=120)


## Parameters

- If you want to use the communication pickle produced by `evaluation/mosta/code/mosta_multilayer_communication_focus_anchor_local.py`, set `COMM_PKL` to that file.
- Important: that PKL must contain the **t3 time key** (usually `3.0` or `E15.5`). If it only contains keys like `0.0/0.5/1.0`, it cannot draw the `t3` plots.


In [ ]:
# Parameters you may edit
from pathlib import Path

RESULT_DIR_NAME = 'mosta_velocity_communication_focus_brain_t3_notebook'

TIMEPOINT_INDEX = 3  # t3
TIMEPOINT_LABELS = ['E12.5', 'E13.5', 'E14.5', 'E15.5']

FOCUS_CELL = 'Brain'
SPACES = ('physical', 'gene')

# Plot look
DENSITY = 2.0
FIGSIZE = (16, 20)
MODE = 'default'

# Save
OUT_DIR = DOWNSTREAM_ROOT / 'results' / RESULT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FORMATS = ('svg',)  # change to ('svg','png') if you also want PNG

# Communication (compute from observed slice; do NOT load a precomputed PKL)
COMM_SAMPLE_N = 20000  # None uses all cells at this timepoint (can be slow / memory heavy)
COMM_RANDOM_SEED = 0
COMM_REMOVE_SELF_LOOP = False
COMM_WINSOR_QUANTILE = 0.995
COMM_DISTANCE_BINS = None
COMM_N_PERMUTATIONS = 0
COMM_PLOT_HEATMAP = False

# Attention extraction
# - save_dense_matrix=False avoids allocating an NxN matrix (O(N^2) memory)
# - save_files=False avoids writing .npy files
ATTN_SAVE_FILES = False
ATTN_SAVE_DENSE_MATRIX = False

# Optional external palette source; leave missing and the notebook will use assets JSON / fallback colors.
COLOR_H5AD = DOWNSTREAM_ROOT / 'spatial_data' / 'Mouse_embryo_all_stage.h5ad'


def _load_palette_from_h5ad(h5ad_path, obs_key_candidates=('Annotation', 'annotation', 'bin_annotation')):
    # Load {category: color} from an h5ad's <obs_key>_colors in .uns (backed='r').
    if h5ad_path is None:
        return None
    h5ad_path = Path(h5ad_path)
    if not h5ad_path.exists():
        print(f'[warn] COLOR_H5AD not found: {h5ad_path}')
        return None

    try:
        adata = sc.read_h5ad(str(h5ad_path), backed='r')
    except Exception as exc:
        print(f'[warn] Failed to read COLOR_H5AD: {exc}')
        return None

    try:
        for obs_key in obs_key_candidates:
            if obs_key not in adata.obs:
                continue
            colors_key = f'{obs_key}_colors'
            if colors_key not in adata.uns:
                continue
            colors = list(adata.uns[colors_key])
            series = adata.obs[obs_key]
            if hasattr(series, 'cat'):
                cats = [str(x) for x in series.cat.categories.tolist()]
            else:
                cats = sorted({str(x) for x in series.astype(str).unique().tolist()})
            if not cats or not colors:
                continue
            if len(colors) < len(cats):
                colors = colors * (len(cats) // len(colors) + 1)
            return {str(cat): str(colors[i]) for i, cat in enumerate(cats)}
    finally:
        try:
            adata.file.close()
        except Exception:
            pass

    print(f'[warn] No palette found in COLOR_H5AD for keys={list(obs_key_candidates)}')
    return None


def _build_palette(categories):
    categories = [str(c) for c in categories]
    if len(categories) <= 20:
        colors = list(sc.pl.palettes.vega_20)
    elif len(categories) <= 28:
        colors = list(sc.pl.palettes.zeileis_28)
    else:
        colors = list(sc.pl.palettes.godsnot_102)
    if len(categories) > len(colors):
        colors = colors * (len(categories) // len(colors) + 1)
    return {cat: colors[i] for i, cat in enumerate(categories)}


def _save_fig_multi(fig, stem: str):
    for ext in SAVE_FORMATS:
        out = OUT_DIR / f'{stem}.{ext}'
        if ext.lower() == 'svg':
            fig.savefig(out, format='svg', bbox_inches='tight')
        else:
            fig.savefig(out, dpi=300, bbox_inches='tight')
        print('Saved:', out)


In [ ]:
# Load data / models from current downstream repo assets

context = load_mosta_context()
df = context.df.copy()
df['samples'] = df['samples'].astype(float)
f_net = context.runtime.f_net
score_net = context.runtime.score_net
device = context.device

# Determine timepoints from the CSV
samples_sorted = sorted(df['samples'].unique().tolist())
if TIMEPOINT_INDEX < 0 or TIMEPOINT_INDEX >= len(samples_sorted):
    raise ValueError(f'TIMEPOINT_INDEX={TIMEPOINT_INDEX} out of range for samples={samples_sorted}')

TIMEPOINT_VALUE = float(samples_sorted[TIMEPOINT_INDEX])
TIMEPOINT_LABEL = TIMEPOINT_LABELS[TIMEPOINT_INDEX] if TIMEPOINT_INDEX < len(TIMEPOINT_LABELS) else str(TIMEPOINT_VALUE)

print('Using timepoint:', {'index': TIMEPOINT_INDEX, 'value': TIMEPOINT_VALUE, 'label': TIMEPOINT_LABEL})
print('Loaded MOSTA runtime from assets | device:', device)

# Palettes (prefer downstream asset JSON, then optional Mouse embryo h5ad, then fallback palette)
annotation_palette = None
label_color_json = Path(context.assets.label_color_json) if context.assets.label_color_json else None
if label_color_json is not None and label_color_json.exists():
    annotation_palette = json.loads(label_color_json.read_text(encoding='utf-8'))
    print(f'Loaded {len(annotation_palette)} colors from label_to_color.json')
elif 'Annotation' in df.columns:
    cats = sorted(df['Annotation'].astype(str).unique().tolist())
    annotation_palette = _build_palette(cats)
    h5ad_palette = _load_palette_from_h5ad(COLOR_H5AD)
    if isinstance(h5ad_palette, dict) and h5ad_palette:
        annotation_palette = {**annotation_palette, **h5ad_palette}
        print(f'Loaded {len(h5ad_palette)} colors from COLOR_H5AD')
    else:
        print('[warn] Using fallback palette (tab20/vega)')

analyzer = VelocityAnalyzer(df, f_net, score_net, dim=int(context.dim))

# Compute communication (attention -> celltype) from observed cells at this timepoint
feature_cols = [f'x{i}' for i in range(1, int(context.dim) + 1)]

df_obs = df[df['samples'] == float(TIMEPOINT_VALUE)].copy()
if 'Annotation' not in df_obs.columns:
    raise ValueError("Expected df['Annotation'] for communication aggregation")

df_obs['Annotation'] = df_obs['Annotation'].astype(str)

# Optional sampling (random sample across ALL cells at this timepoint)
if COMM_SAMPLE_N is not None and len(df_obs) > int(COMM_SAMPLE_N):
    df_obs = df_obs.sample(n=int(COMM_SAMPLE_N), random_state=int(COMM_RANDOM_SEED)).reset_index(drop=True)

print('Comm slice rows:', len(df_obs), '| focus rows:', int((df_obs['Annotation'] == str(FOCUS_CELL)).sum()))

X_obs = df_obs[feature_cols].to_numpy(dtype=np.float32)
adata_comm = ad.AnnData(X=X_obs)
adata_comm.obs['Annotation'] = df_obs['Annotation'].to_numpy()
adata_comm.obsm['spatial'] = X_obs[:, :2]

attn_out = helpers.save_interpolated_attention(
    adata_comm,
    time_value=float(TIMEPOINT_VALUE),
    f_net=f_net,
    device=device,
    out_dir=None,
    save_files=bool(ATTN_SAVE_FILES),
    save_dense_matrix=bool(ATTN_SAVE_DENSE_MATRIX),
)

comm_one = helpers.analyze_attention_by_celltype(
    edge_index=attn_out['edge_index'],
    attn=attn_out['attn_mean'],
    labels=adata_comm.obs['Annotation'].values,
    spatial_coord=adata_comm.obsm['spatial'],
    time_title=TIMEPOINT_LABEL,
    remove_self_loop=bool(COMM_REMOVE_SELF_LOOP),
    winsor_quantile=COMM_WINSOR_QUANTILE,
    distance_bins=COMM_DISTANCE_BINS,
    n_permutations=int(COMM_N_PERMUTATIONS),
    plot=bool(COMM_PLOT_HEATMAP),
)

comm = {TIMEPOINT_LABEL: comm_one}
COMM_TIME_KEY = TIMEPOINT_LABEL
print('COMM_TIME_KEY:', COMM_TIME_KEY)
print('Comm types:', len(comm_one.get('types', [])))


In [ ]:
# Plot the four figures: physical/gene x intrinsic/interaction

for space in SPACES:
    if space not in {'physical', 'gene'}:
        raise ValueError(space)

    adata_res, axes, figs = analyzer.plot_fingerprint(
        adata=None,
        timepoint=TIMEPOINT_VALUE,
        timepoint_str=COMM_TIME_KEY,  # Used for communication lookup and titling
        space=space,
        color='Annotation',
        mode=MODE,
        cell_type=FOCUS_CELL,  # Focus communication edges on Brain
        background_cell_type=None,  # Keep all cells in the velocity background
        communication=True,
        comm_edge_threshold=0.0,
        comm_edge_top_k=3,
        comm_edge_top_k_focus_label=FOCUS_CELL,
        comm_centroid_top_n_y=200,
        comm_centroid_top_n_y_exclude_types=(FOCUS_CELL, 'Meninges','Choroid plexus'),
        figsize=FIGSIZE,
        all_time_communication=comm,
        save_path=None,
        label_to_color=annotation_palette,
        density=DENSITY,
        interaction_m=1024,
        interaction_threshold=1000,
        device=device,
    )

    stem_base = f'velocity_communication_{space}_brain_t{TIMEPOINT_INDEX}_combined_focus-{FOCUS_CELL}'

    _save_fig_multi(figs[0], f'{stem_base}_intrinsic')
    _save_fig_multi(figs[1], f'{stem_base}_interaction')

    plt.show()
    for fig in figs:
        plt.close(fig)


## Notes / troubleshooting

- The output SVGs are written under `results/mosta_velocity_communication_focus_brain_t3_notebook/`.
- This migrated notebook computes the communication matrix from the observed slice directly and does not depend on any precomputed PKL outside the current repo.
